# Regression - Continuous Price Prediction (Day 10)

## Objective
Predict **exact EUR/MWh price** (continuous value) instead of binary high/low classification.

## Key Differences from Classification
* **Target**: `price_eur_mwh` (continuous) vs `high_price_period` (binary)
* **Models**: Linear Regression, Random Forest Regressor, GBT Regressor
* **Metrics**: RMSE, MAE, R² vs AUC, F1, Precision
* **Business Value**: Precise cost forecasting for H2 production planning

## Components
1. **Linear Regression**: Baseline model with interpretable coefficients
2. **Random Forest Regressor**: Ensemble method for non-linear relationships
3. **Gradient Boosted Trees**: Sequential ensemble for complex patterns
4. **Evaluation**: Compare models with regression metrics
5. **Gold Layer**: Save predictions to `h2_gold_regression_predictions`

## Expected Outcomes
* **RMSE**: Target < 20 EUR/MWh (better than naive mean prediction)
* **R²**: Target > 0.60 (explain 60%+ of variance)
* **MAE**: Target < 15 EUR/MWh (avg error within 15 EUR)

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
import mlflow
import mlflow.spark
from datetime import datetime

print("✅ Imports loaded")

In [0]:
# Data configuration
CATALOG = "dbacademy"
SCHEMA = "labuser13955035_1772775399"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.h2_gold_model_scoring_base"

# Target column (continuous value)
TARGET_COL = "price_eur_mwh"
TIME_COL = "event_time_utc"
SPLIT_DATE = "2021-01-01"

# Output table
REGRESSION_PREDICTIONS_TABLE = f"{CATALOG}.{SCHEMA}.h2_gold_regression_predictions"
REGRESSION_LEADERBOARD_TABLE = f"{CATALOG}.{SCHEMA}.h2_gold_regression_leaderboard"

# MLflow experiment
EXPERIMENT_NAME = "/Users/labuser13955035_1772775399@vocareum.com/H2_Price_Regression"

print(f"Source: {SOURCE_TABLE}")
print(f"Target: {TARGET_COL}")
print(f"Output predictions: {REGRESSION_PREDICTIONS_TABLE}")
print(f"Output leaderboard: {REGRESSION_LEADERBOARD_TABLE}")
print(f"MLflow Experiment: {EXPERIMENT_NAME}")

In [0]:
print("Loading data...")
print("="*80)

# Load source data
df = spark.table(SOURCE_TABLE)

print(f"Total records: {df.count():,}")

# Split by time (2020 = train, 2021 = test)
train = df.filter(F.col(TIME_COL) < SPLIT_DATE)
test = df.filter(F.col(TIME_COL) >= SPLIT_DATE)

train_count = train.count()
test_count = test.count()

print(f"\nTrain (2020): {train_count:,} records")
print(f"Test (2021):  {test_count:,} records")

# Show price statistics
print(f"\nPrice Statistics:")
train_stats = train.select(TARGET_COL).summary("min", "25%", "50%", "75%", "max", "mean", "stddev")
display(train_stats)

print("✅ Data loaded and split")

In [0]:
# Define feature columns (same as classification)
feature_cols = [
    "avg_actual_total_load_mw",
    "day_ahead_total_load_forecast_mw",
    "temperature_c",
    "u10_ms",
    "v10_ms",
    "wind_speed_ms",
    "surface_pressure_pa",
    "renewable_generation_mw",
    "non_renewable_generation_mw",
    "renewable_share_pct",
    "non_renewable_share_pct",
    "hour_of_day",
    "day_of_week",
    "month"
]

print(f"Features: {len(feature_cols)} columns")
for col in feature_cols:
    print(f"  - {col}")

# Vector assembler
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

print("\n✅ Feature assembler ready")

In [0]:
print("Training Linear Regression...")
print("="*80)

# Set MLflow experiment
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name="linear_regression") as run:
    # Model
    lr = LinearRegression(
        featuresCol="features",
        labelCol=TARGET_COL,
        maxIter=100,
        regParam=0.01,
        elasticNetParam=0.5
    )
    
    # Pipeline
    pipeline_lr = Pipeline(stages=[assembler, lr])
    
    # Train
    import time
    start_time = time.time()
    model_lr = pipeline_lr.fit(train)
    training_time = time.time() - start_time
    
    # Predictions
    train_pred_lr = model_lr.transform(train)
    test_pred_lr = model_lr.transform(test)
    
    # Evaluate
    evaluator_rmse = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="rmse")
    evaluator_mae = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="mae")
    evaluator_r2 = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="r2")
    
    train_rmse = evaluator_rmse.evaluate(train_pred_lr)
    train_mae = evaluator_mae.evaluate(train_pred_lr)
    train_r2 = evaluator_r2.evaluate(train_pred_lr)
    
    test_rmse = evaluator_rmse.evaluate(test_pred_lr)
    test_mae = evaluator_mae.evaluate(test_pred_lr)
    test_r2 = evaluator_r2.evaluate(test_pred_lr)
    
    # Log to MLflow
    mlflow.log_param("model_type", "linear_regression")
    mlflow.log_param("max_iter", 100)
    mlflow.log_param("reg_param", 0.01)
    mlflow.log_param("elastic_net_param", 0.5)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_r2", test_r2)
    mlflow.log_metric("training_time_sec", training_time)
    mlflow.spark.log_model(model_lr, "model")
    
    print(f"Linear Regression Results:")
    print(f"  Train RMSE: {train_rmse:.2f} EUR/MWh")
    print(f"  Test RMSE:  {test_rmse:.2f} EUR/MWh")
    print(f"  Train MAE:  {train_mae:.2f} EUR/MWh")
    print(f"  Test MAE:   {test_mae:.2f} EUR/MWh")
    print(f"  Train R²:   {train_r2:.4f}")
    print(f"  Test R²:    {test_r2:.4f}")
    print(f"  Training:   {training_time:.2f}s")
    print(f"\n✅ Linear Regression trained")

In [0]:
print("Training Random Forest Regressor...")
print("="*80)

with mlflow.start_run(run_name="random_forest_regressor") as run:
    # Model
    rf = RandomForestRegressor(
        featuresCol="features",
        labelCol=TARGET_COL,
        numTrees=100,
        maxDepth=12,
        seed=42
    )
    
    # Pipeline
    pipeline_rf = Pipeline(stages=[assembler, rf])
    
    # Train
    start_time = time.time()
    model_rf = pipeline_rf.fit(train)
    training_time = time.time() - start_time
    
    # Predictions
    train_pred_rf = model_rf.transform(train)
    test_pred_rf = model_rf.transform(test)
    
    # Evaluate
    train_rmse = evaluator_rmse.evaluate(train_pred_rf)
    train_mae = evaluator_mae.evaluate(train_pred_rf)
    train_r2 = evaluator_r2.evaluate(train_pred_rf)
    
    test_rmse = evaluator_rmse.evaluate(test_pred_rf)
    test_mae = evaluator_mae.evaluate(test_pred_rf)
    test_r2 = evaluator_r2.evaluate(test_pred_rf)
    
    # Log to MLflow
    mlflow.log_param("model_type", "random_forest_regressor")
    mlflow.log_param("num_trees", 100)
    mlflow.log_param("max_depth", 12)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_r2", test_r2)
    mlflow.log_metric("training_time_sec", training_time)
    mlflow.spark.log_model(model_rf, "model")
    
    print(f"Random Forest Results:")
    print(f"  Train RMSE: {train_rmse:.2f} EUR/MWh")
    print(f"  Test RMSE:  {test_rmse:.2f} EUR/MWh")
    print(f"  Train MAE:  {train_mae:.2f} EUR/MWh")
    print(f"  Test MAE:   {test_mae:.2f} EUR/MWh")
    print(f"  Train R²:   {train_r2:.4f}")
    print(f"  Test R²:    {test_r2:.4f}")
    print(f"  Training:   {training_time:.2f}s")
    print(f"\n✅ Random Forest trained")

In [0]:
print("Training Gradient Boosted Trees Regressor...")
print("="*80)
print("⚠️  Using reduced parameters for faster training on single-node cluster")

with mlflow.start_run(run_name="gbt_regressor") as run:
    # Model (reduced params)
    gbt = GBTRegressor(
        featuresCol="features",
        labelCol=TARGET_COL,
        maxIter=50,   # Reduced from 100
        maxDepth=5,   # Reduced from 10
        stepSize=0.1,
        seed=42
    )
    
    # Pipeline
    pipeline_gbt = Pipeline(stages=[assembler, gbt])
    
    # Train
    start_time = time.time()
    model_gbt = pipeline_gbt.fit(train)
    training_time = time.time() - start_time
    
    # Predictions
    train_pred_gbt = model_gbt.transform(train)
    test_pred_gbt = model_gbt.transform(test)
    
    # Evaluate
    train_rmse = evaluator_rmse.evaluate(train_pred_gbt)
    train_mae = evaluator_mae.evaluate(train_pred_gbt)
    train_r2 = evaluator_r2.evaluate(train_pred_gbt)
    
    test_rmse = evaluator_rmse.evaluate(test_pred_gbt)
    test_mae = evaluator_mae.evaluate(test_pred_gbt)
    test_r2 = evaluator_r2.evaluate(test_pred_gbt)
    
    # Log to MLflow
    mlflow.log_param("model_type", "gbt_regressor")
    mlflow.log_param("max_iter", 50)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("step_size", 0.1)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_r2", test_r2)
    mlflow.log_metric("training_time_sec", training_time)
    mlflow.spark.log_model(model_gbt, "model")
    
    print(f"GBT Regressor Results:")
    print(f"  Train RMSE: {train_rmse:.2f} EUR/MWh")
    print(f"  Test RMSE:  {test_rmse:.2f} EUR/MWh")
    print(f"  Train MAE:  {train_mae:.2f} EUR/MWh")
    print(f"  Test MAE:   {test_mae:.2f} EUR/MWh")
    print(f"  Train R²:   {train_r2:.4f}")
    print(f"  Test R²:    {test_r2:.4f}")
    print(f"  Training:   {training_time:.2f}s")
    print(f"\n✅ GBT Regressor trained")

In [0]:
print("="*80)
print("REGRESSION MODEL LEADERBOARD (sorted by Test RMSE)")
print("="*80)

# Get all runs from experiment
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# Select key columns
leaderboard = runs[[
    "tags.mlflow.runName",
    "metrics.test_rmse",
    "metrics.test_mae",
    "metrics.test_r2",
    "metrics.training_time_sec"
]].rename(columns={
    "tags.mlflow.runName": "model_name",
    "metrics.test_rmse": "test_rmse",
    "metrics.test_mae": "test_mae",
    "metrics.test_r2": "test_r2",
    "metrics.training_time_sec": "training_time_sec"
}).sort_values("test_rmse")

display(leaderboard)

# Best model
best_model_name = leaderboard.iloc[0]["model_name"]
best_rmse = leaderboard.iloc[0]["test_rmse"]
best_mae = leaderboard.iloc[0]["test_mae"]
best_r2 = leaderboard.iloc[0]["test_r2"]

print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"\nPerformance:")
print(f"  Test RMSE: {best_rmse:.2f} EUR/MWh")
print(f"  Test MAE:  {best_mae:.2f} EUR/MWh")
print(f"  Test R²:   {best_r2:.4f}")

# Save leaderboard to gold layer
leaderboard_spark = spark.createDataFrame(leaderboard)
leaderboard_spark.write.mode("overwrite").saveAsTable(REGRESSION_LEADERBOARD_TABLE)
print(f"\n✅ Leaderboard saved to: {REGRESSION_LEADERBOARD_TABLE}")

In [0]:
print("Saving best model predictions...")
print("="*80)

# Determine which model was best
if best_model_name == "linear_regression":
    best_predictions = test_pred_lr
elif best_model_name == "random_forest_regressor":
    best_predictions = test_pred_rf
else:
    best_predictions = test_pred_gbt

# Select relevant columns
predictions_to_save = best_predictions.select(
    TIME_COL,
    "zone",
    F.col(TARGET_COL).alias("actual_price_eur_mwh"),
    F.col("prediction").alias("predicted_price_eur_mwh"),
    (F.col("prediction") - F.col(TARGET_COL)).alias("error_eur_mwh"),
    F.abs(F.col("prediction") - F.col(TARGET_COL)).alias("abs_error_eur_mwh")
).withColumn(
    "prediction_timestamp",
    F.current_timestamp()
).withColumn(
    "model_name",
    F.lit(best_model_name)
)

# Save to gold layer
predictions_to_save.write.mode("overwrite").saveAsTable(REGRESSION_PREDICTIONS_TABLE)

record_count = spark.table(REGRESSION_PREDICTIONS_TABLE).count()
print(f"✅ Predictions saved: {record_count:,} records")
print(f"   Table: {REGRESSION_PREDICTIONS_TABLE}")

# Show sample predictions
print(f"\nSample Predictions (sorted by absolute error):")
display(predictions_to_save.orderBy(F.desc("abs_error_eur_mwh")).limit(20))

In [0]:
print("Error Distribution Analysis")
print("="*80)

# Load saved predictions
preds = spark.table(REGRESSION_PREDICTIONS_TABLE)

# Overall statistics
print("\nOverall Error Statistics:")
error_stats = preds.select(
    F.mean("error_eur_mwh").alias("mean_error"),
    F.stddev("error_eur_mwh").alias("stddev_error"),
    F.mean("abs_error_eur_mwh").alias("mae"),
    F.percentile_approx("abs_error_eur_mwh", 0.5).alias("median_abs_error"),
    F.percentile_approx("abs_error_eur_mwh", 0.95).alias("95th_percentile_error")
).collect()[0]

print(f"  Mean Error:        {error_stats['mean_error']:.2f} EUR/MWh (bias)")
print(f"  Std Dev Error:     {error_stats['stddev_error']:.2f} EUR/MWh")
print(f"  MAE:               {error_stats['mae']:.2f} EUR/MWh")
print(f"  Median Abs Error:  {error_stats['median_abs_error']:.2f} EUR/MWh")
print(f"  95th Percentile:   {error_stats['95th_percentile_error']:.2f} EUR/MWh")

# Error by hour of day
print(f"\nError by Hour of Day:")
error_by_hour = preds.withColumn(
    "hour",
    F.hour(TIME_COL)
).groupBy("hour").agg(
    F.mean("abs_error_eur_mwh").alias("avg_abs_error"),
    F.count("*").alias("count")
).orderBy("hour")

display(error_by_hour)

print("✅ Error analysis complete")

## ✅ Regression Model Complete!

### What We Built
1. **Linear Regression**: Baseline model with interpretable coefficients
2. **Random Forest Regressor**: Ensemble method capturing non-linearities
3. **GBT Regressor**: Sequential boosting (reduced params for speed)
4. **MLflow Tracking**: All experiments logged with metrics
5. **Gold Tables**: Predictions and leaderboard saved

### Key Findings
* **Best Model**: Determined by lowest Test RMSE
* **Metrics Comparison**: RMSE, MAE, R² across all models
* **Error Analysis**: Bias, variance, and error patterns by hour

### Business Value
* **Precise Forecasting**: Predict exact EUR/MWh prices (not just high/low)
* **Cost Planning**: Accurate cost estimates for H2 production scheduling
* **Error Bounds**: Understand prediction uncertainty with MAE/RMSE
* **Hourly Patterns**: Identify times of day with higher/lower forecast accuracy

### Comparison: Classification vs Regression

| Aspect | Classification (Days 5-9) | Regression (Day 10) |
| --- | --- | --- |
| **Target** | Binary (high/low price) | Continuous (EUR/MWh) |
| **Metrics** | AUC, F1, Precision, Recall | RMSE, MAE, R² |
| **Use Case** | Risk alerts & production windows | Precise cost forecasting |
| **Complexity** | Simpler (2 classes) | More challenging (infinite values) |
| **Output** | Go/No-Go decisions | Exact price predictions |

### Next Steps
* **Day 11**: Time series forecasting (Prophet, ARIMA)
* **Day 12**: AutoML for automated model selection
* **Day 13**: Advanced feature engineering & hyperparameter tuning
* **Day 14**: Model ensemble & final evaluation

### Deployment Considerations
* Monitor prediction drift (RMSE over time)
* Retrain monthly with new data
* Set up alerts for high-error predictions (> 2× MAE)
* A/B test regression vs classification in production